# Polyp Detection — Data Exploration

This notebook extracts and explores both datasets before any
training takes place. The goal is to understand the data
distribution, verify annotation quality, and identify any
potential issues (missing masks, resolution variance, polyp
size distribution) that might affect training decisions.

---

## What this notebook does

1. Extracts Kvasir-SEG and CVC-ClinicDB from zip files
2. Verifies dataset structure and image-mask pairing
3. Visualizes sample images with their masks and overlays
4. Computes and compares statistics across both datasets

## Datasets

- Kvasir-SEG: 1,000 colonoscopy images — train/val/test split
- CVC-ClinicDB: 612 colonoscopy images — cross-dataset evaluation only

## Output

- results/figures/kvasir_samples.png
- results/figures/cvc_samples.png
- results/figures/dataset_comparison.png

In [ ]:
# Import libraries

import os
import random
import sys
import time
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

In [ ]:
# Config & Paths
# Project paths and settings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    # Raw zip files
    "kvasir_zip": os.path.join(BASE_DIR, "data", "raw", "kvasir-seg.zip"),
    "cvc_zip":    os.path.join(BASE_DIR, "data", "raw", "cvc-clinicdb.zip"),

    # Extracted dataset folders
    "kvasir_dir": os.path.join(BASE_DIR, "data", "kvasir-seg"),
    "cvc_dir":    os.path.join(BASE_DIR, "data", "cvc-clinicdb"),

    # Output folders
    "figures":    os.path.join(BASE_DIR, "results", "figures"),
    "metrics":    os.path.join(BASE_DIR, "results", "metrics"),
}

for key in ["figures", "metrics"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:12s} -> {path}")

In [ ]:
# Extract Datasets
# Run this once to extract zip files into their target folders

def extract_zip(zip_path, target_dir, dataset_name):
    if not os.path.exists(zip_path):
        print(f"ERROR: zip not found -> {zip_path}")
        return False

    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"{dataset_name}: already extracted, skipping.")
        return True

    print(f"Extracting {dataset_name}...")
    os.makedirs(target_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target_dir)

    print(f"{dataset_name} extracted to: {target_dir}")
    return True


extract_zip(PATHS["kvasir_zip"], PATHS["kvasir_dir"], "Kvasir-SEG")
extract_zip(PATHS["cvc_zip"],    PATHS["cvc_dir"],    "CVC-ClinicDB")

In [ ]:
# Inspect Raw Structure
# Check what folders came out after extraction

def print_structure(root_dir, dataset_name, max_depth=3):
    print(f"\n{dataset_name} structure:")
    print("-" * 40)

    for dirpath, dirnames, filenames in os.walk(root_dir):
        depth = dirpath.replace(root_dir, "").count(os.sep)
        if depth >= max_depth:
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath)}/")

        if depth == max_depth - 1 and filenames:
            sub   = "  " * (depth + 1)
            shown = filenames[:3]
            for f in shown:
                print(f"{sub}{f}")
            if len(filenames) > 3:
                print(f"{sub}... ({len(filenames)} files total)")


print_structure(PATHS["kvasir_dir"], "Kvasir-SEG")
print_structure(PATHS["cvc_dir"],    "CVC-ClinicDB")

In [ ]:
# Locate Images and Masks
# Handles different naming conventions across datasets
# (e.g. "images"/"masks" vs "Original"/"Ground Truth")

def find_image_mask_dirs(root_dir):
    """Walk the directory tree to find image and mask folders."""
    image_names = {"images", "original"}
    mask_names  = {"masks", "ground truth", "ground_truth"}

    images_dir = None
    masks_dir  = None

    for dirpath, dirnames, _ in os.walk(root_dir):
        basename = os.path.basename(dirpath).lower()

        # Skip the TIF variant if a PNG version exists - we prefer PNG
        if "tif" in dirpath.lower():
            continue

        if basename in image_names and images_dir is None:
            images_dir = dirpath
        if basename in mask_names and masks_dir is None:
            masks_dir = dirpath

    return images_dir, masks_dir


def get_file_list(folder, extensions=(".jpg", ".jpeg", ".png", ".tif", ".tiff")):
    if folder is None or not os.path.exists(folder):
        return []
    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(extensions)
    ])


kvasir_img_dir, kvasir_mask_dir = find_image_mask_dirs(PATHS["kvasir_dir"])
cvc_img_dir,    cvc_mask_dir    = find_image_mask_dirs(PATHS["cvc_dir"])

kvasir_images = get_file_list(kvasir_img_dir)
kvasir_masks  = get_file_list(kvasir_mask_dir)
cvc_images    = get_file_list(cvc_img_dir)
cvc_masks     = get_file_list(cvc_mask_dir)

print("DATASET SUMMARY")
print("-" * 40)
print(f"Kvasir-SEG:   images={len(kvasir_images)}  masks={len(kvasir_masks)}")
print(f"CVC-ClinicDB: images={len(cvc_images)}  masks={len(cvc_masks)}")
print()
print(f"Kvasir images dir: {kvasir_img_dir}")
print(f"Kvasir masks dir:  {kvasir_mask_dir}")
print(f"CVC images dir:    {cvc_img_dir}")
print(f"CVC masks dir:     {cvc_mask_dir}")

In [ ]:
# Verify Image-Mask Pairing
# Check that every image has a corresponding mask
# Mismatches here would cause silent errors during training

def verify_pairing(images, masks, dataset_name):
    print(f"\n{dataset_name} - pairing check:")

    img_stems  = {os.path.splitext(f)[0] for f in images}
    mask_stems = {os.path.splitext(f)[0] for f in masks}

    missing_masks  = img_stems  - mask_stems
    missing_images = mask_stems - img_stems

    if not missing_masks and not missing_images:
        print(f"  All {len(images)} images have a matching mask.")
    else:
        if missing_masks:
            print(f"  WARNING: {len(missing_masks)} images have no mask: {list(missing_masks)[:5]}")
        if missing_images:
            print(f"  WARNING: {len(missing_images)} masks have no image: {list(missing_images)[:5]}")


verify_pairing(kvasir_images, kvasir_masks, "Kvasir-SEG")
verify_pairing(cvc_images,    cvc_masks,    "CVC-ClinicDB")

In [ ]:
# Visual Exploration
# Display sample image / mask / overlay triplets
# Overlay helps verify that masks are correctly aligned with images

def find_mask_file(masks_dir, image_stem):
    """Find a mask file matching the image stem, regardless of extension."""
    for ext in (".png", ".jpg", ".jpeg", ".tif", ".tiff"):
        candidate = os.path.join(masks_dir, image_stem + ext)
        if os.path.exists(candidate):
            return candidate
    return None


def load_rgb(path):
    img = cv2.imread(path)
    if img is None:
        img = np.array(Image.open(path).convert("RGB"))
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


def load_mask(path):
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        mask = np.array(Image.open(path).convert("L"))
    return mask


def make_overlay(image, mask, alpha=0.4):
    """Blend a green mask on top of the image."""
    overlay         = image.copy().astype(np.float32)
    binary          = mask > 127
    green           = np.array([0, 200, 0], dtype=np.float32)
    overlay[binary] = overlay[binary] * (1 - alpha) + green * alpha
    return overlay.astype(np.uint8)


def show_samples(img_dir, mask_dir, file_list, title, n=5, save_path=None):
    samples = random.sample(file_list, min(n, len(file_list)))

    fig, axes = plt.subplots(n, 3, figsize=(13, n * 4))
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)

    col_titles = ["Image", "Mask", "Overlay"]
    for ax, ct in zip(axes[0], col_titles):
        ax.set_title(ct, fontsize=11, fontweight="bold")

    for i, fname in enumerate(samples):
        stem      = os.path.splitext(fname)[0]
        img_path  = os.path.join(img_dir, fname)
        mask_path = find_mask_file(mask_dir, stem)

        if mask_path is None:
            print(f"WARNING: no mask for {fname}")
            continue

        img     = load_rgb(img_path)
        mask    = load_mask(mask_path)
        overlay = make_overlay(img, mask)

        for j, data in enumerate([img, mask, overlay]):
            ax = axes[i][j]
            if j == 1:
                ax.imshow(data, cmap="gray")
            else:
                ax.imshow(data)
            ax.axis("off")

        axes[i][0].set_ylabel(stem[:20], fontsize=8, rotation=0,
                              labelpad=60, va="center")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"Saved -> {save_path}")

    plt.show()


show_samples(
    kvasir_img_dir, kvasir_mask_dir, kvasir_images,
    title="Kvasir-SEG - Sample Images",
    save_path=os.path.join(PATHS["figures"], "kvasir_samples.png")
)

show_samples(
    cvc_img_dir, cvc_mask_dir, cvc_images,
    title="CVC-ClinicDB - Sample Images",
    save_path=os.path.join(PATHS["figures"], "cvc_samples.png")
)

In [ ]:
# Dataset Statistics
# Compute resolution and polyp area statistics for both datasets
# Polyp area ratio = what fraction of each image is covered by a polyp
# This tells us how "hard" the dataset is - smaller polyps are harder to detect

def compute_stats(img_dir, mask_dir, file_list, dataset_name):
    widths, heights, polyp_ratios = [], [], []

    for fname in file_list:
        stem = os.path.splitext(fname)[0]

        img  = Image.open(os.path.join(img_dir, fname))
        w, h = img.size
        widths.append(w)
        heights.append(h)

        mask_path = find_mask_file(mask_dir, stem)
        if mask_path:
            mask  = load_mask(mask_path)
            ratio = (mask > 127).sum() / mask.size
            polyp_ratios.append(ratio)

    print(f"\n{dataset_name}")
    print("-" * 40)
    print(f"Total images:   {len(file_list)}")
    print(f"Width range:    {min(widths)}-{max(widths)} px (avg {np.mean(widths):.0f})")
    print(f"Height range:   {min(heights)}-{max(heights)} px (avg {np.mean(heights):.0f})")
    print(f"Polyp area min: {min(polyp_ratios)*100:.1f}%")
    print(f"Polyp area max: {max(polyp_ratios)*100:.1f}%")
    print(f"Polyp area avg: {np.mean(polyp_ratios)*100:.1f}%")
    print(f"Polyp area med: {np.median(polyp_ratios)*100:.1f}%")

    return widths, heights, polyp_ratios


kvasir_w, kvasir_h, kvasir_ratios = compute_stats(
    kvasir_img_dir, kvasir_mask_dir, kvasir_images, "Kvasir-SEG"
)
cvc_w, cvc_h, cvc_ratios = compute_stats(
    cvc_img_dir, cvc_mask_dir, cvc_images, "CVC-ClinicDB"
)

In [ ]:
# Comparison Plot
# Side-by-side comparison of both datasets

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Dataset Comparison: Kvasir-SEG vs CVC-ClinicDB",
             fontsize=13, fontweight="bold")

# Polyp area distribution
axes[0].hist(kvasir_ratios, bins=25, alpha=0.7,
             color="steelblue", edgecolor="white", label="Kvasir-SEG")
axes[0].hist(cvc_ratios, bins=25, alpha=0.7,
             color="coral", edgecolor="white", label="CVC-ClinicDB")
axes[0].axvline(np.mean(kvasir_ratios), color="steelblue",
                linestyle="--", linewidth=1.5)
axes[0].axvline(np.mean(cvc_ratios), color="coral",
                linestyle="--", linewidth=1.5)
axes[0].set_title("Polyp Area Distribution")
axes[0].set_xlabel("Fraction of image covered by polyp")
axes[0].set_ylabel("Number of images")
axes[0].legend()

# Image width comparison
axes[1].scatter(range(len(kvasir_w)), kvasir_w,
                alpha=0.4, s=8, color="steelblue", label="Kvasir-SEG")
axes[1].scatter(range(len(cvc_w)), cvc_w,
                alpha=0.4, s=8, color="coral", label="CVC-ClinicDB")
axes[1].set_title("Image Widths")
axes[1].set_xlabel("Image index")
axes[1].set_ylabel("Width (px)")
axes[1].legend()

# Dataset size comparison
counts     = [len(kvasir_images), len(cvc_images)]
labels     = ["Kvasir-SEG\n(train/val/test)", "CVC-ClinicDB\n(cross-dataset test)"]
bar_colors = ["steelblue", "coral"]
bars = axes[2].bar(labels, counts, color=bar_colors, edgecolor="white", width=0.5)
axes[2].set_title("Dataset Sizes")
axes[2].set_ylabel("Number of images")
for bar, count in zip(bars, counts):
    axes[2].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 10,
                 str(count), ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "dataset_comparison.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Summary

print("NOTEBOOK 01 COMPLETE")
print("=" * 40)
print(f"Kvasir-SEG:   {len(kvasir_images)} images, {len(kvasir_masks)} masks")
print(f"CVC-ClinicDB: {len(cvc_images)} images, {len(cvc_masks)} masks")
print()
print("Figures saved:")
for fname in ["kvasir_samples.png", "cvc_samples.png", "dataset_comparison.png"]:
    path   = os.path.join(PATHS["figures"], fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
print()
print("Next -> 02_data_preparation.ipynb")
print("  - Convert masks to YOLO bounding boxes")
print("  - Stratified train/val/test split")
print("  - Verify YOLO format")